# Gap Parameter Fitting — Eq. C1 Method

This notebook fits a parametric brightness temperature model (Eq. C1 from the paper appendix) to **frank radial profiles** of each disk, in order to measure the center, width, and depth of each gap of interest.

The resulting parameters (`rgap`, `σgap`, `δgap`, `T0`, `q`) are then used to define the injection zone in the injection-recovery pipeline (`HPC_eqC1_gap/`).

## The Model — Equation C1

The brightness temperature profile is modelled as a **local power-law background with a Gaussian depletion**:

$$T_b(r) = T_0 \left(\frac{r}{0.1''}\right)^{-q} \cdot \frac{1}{1 + \Gamma}$$

where the depletion term is:

$$\Gamma(r) = (\delta_\mathrm{gap} - 1) \cdot \exp\!\left[-\frac{(r - r_\mathrm{gap})^2}{2\,\sigma_\mathrm{gap}^2}\right]$$

### Parameters

All 5 parameters are **free parameters fitted from the frank radial profile** — you do not need to know any of them beforehand. You only need to provide:
- The frank radial profile (the data)
- A radial range `[r_min, r_max]` that brackets the gap on both sides
- Rough initial guesses (read off by eye from the profile plot)

| Parameter | Symbol | What it describes | How to guess |
|---|---|---|---|
| Background normalisation | $T_0$ | T_b value at r = 0.1" on the local power-law (K) | Read T_b near the gap from the frank plot |
| Power-law index | $q$ | How steeply the background falls with radius | Start with q ≈ 1–3; steeper disk → higher q |
| Gap centre | $r_\mathrm{gap}$ | Radial location of the gap minimum (arcsec) | Read off from the frank profile dip |
| Gap width | $\sigma_\mathrm{gap}$ | Gaussian std dev of the gap (arcsec) | Roughly half the visible gap width |
| Gap depth | $\delta_\mathrm{gap}$ | Multiplicative depletion at $r_\mathrm{gap}$ | Ratio of background to gap floor; start with 2–10 |

**Note on δgap**: multiplicative scale, not fractional depth.
- δgap = 1 → no gap (Γ = 0)
- δgap = 2 → brightness halved at gap centre
- δgap = 10 → order-of-magnitude depletion at gap centre

**Note on T0, q**: describe the *local* background near the gap only — fit a narrow radial window, not the entire disk.

## Workflow

1. Load the **frank radial profile** for the disk (T_b vs r in arcsec)
2. Isolate the radial range around the gap of interest
3. Fit Eq. C1 to that sub-profile using `scipy.optimize.curve_fit`
4. Visually inspect the fit — adjust initial guesses if needed
5. Record the fitted parameters in `diskdictionary_eqC1.py`

### What the frank visibility comparison is (not needed here)

The paper (Figure 14) also shows deprojected, azimuthally averaged visibilities vs the frank model as a **quality check that frank converged correctly** in the UV/Fourier domain. This is already done when you ran frank — you do not need to redo it here. The Eq. C1 fitting is done entirely on the frank output radial profile in image space.

## Adopted Parameters Table

To be filled in as fits are completed. Mirrors Table 3 in the paper.

| Disk | Gap | $T_0$ (K) | $q$ | $r_\mathrm{gap}$ (") | $\sigma_\mathrm{gap}$ (") | $\delta_\mathrm{gap}$ |
|---|---|---|---|---|---|---|
| J1852 | 0 | | | | | |
| AA_Tau | 0 | | | | | |
| AA_Tau | 1 | | | | | |
| CQ_Tau | 0 | | | | | |
| DM_Tau | 0 | | | | | |
| DM_Tau | 1 | | | | | |
| HD_135344B | 0 | | | | | |
| HD_135344B | 1 | | | | | |
| HD_143006 | 0 | | | | | |
| HD_34282 | 0 | | | | | |
| J1604 | 0 | | | | | |
| J1615 | 0 | | | | | |
| J1615 | 1 | | | | | |
| J1842 | 0 | | | | | |
| LkCa_15 | 0 | | | | | |
| LkCa_15 | 1 | | | | | |
| MWC_758 | 0 | | | | | |
| MWC_758 | 1 | | | | | |
| SY_Cha | 0 | | | | | |

---
## Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit

In [ ]:
# ── Load gap_clicks.csv — manual click estimates per disk/gap ────────────────
clicks = pd.read_csv(r'D:\CPD_MPIA\Injection_Recovery_Revised\gap_clicks.csv')

def get_gap_guesses(disk, gap_ix):
    """Return (r_min, r_max, r_gap_guess, sigma_gap_guess) from gap_clicks.
    
    r_min  = inner_edge - 0.05"  (a little inside, to catch background)
    r_max  = ring_peak  + 0.05"  (a little outside, to catch background)
    r_gap  = gap_dip             (the clicked gap minimum)
    sigma  = (outer_edge - inner_edge) / 2  (rough half-width)
    """
    d = clicks[clicks['disk'] == disk].copy()
    # each gap is a group of 4 rows (dip, ring_peak, inner_edge, outer_edge)
    # gap_ix=0 → first group, gap_ix=1 → second group
    dips        = d[d['label'] == 'gap_dip'      ].reset_index(drop=True)
    rings       = d[d['label'] == 'ring_peak'    ].reset_index(drop=True)
    inner_edges = d[d['label'] == 'inner_edge'   ].reset_index(drop=True)
    outer_edges = d[d['label'] == 'outer_edge'   ].reset_index(drop=True)

    if gap_ix >= len(dips):
        raise ValueError(f'gap_ix={gap_ix} not found for {disk} in gap_clicks.csv')

    r_dip   = dips.loc[gap_ix,   'r_arcsec']
    r_ring  = rings.loc[gap_ix,  'r_arcsec']
    r_inner = inner_edges.loc[gap_ix, 'r_arcsec']
    r_outer = outer_edges.loc[gap_ix, 'r_arcsec']

    r_min        = max(r_inner - 0.05, 0.01)
    r_max        = r_ring  + 0.05
    r_gap_guess  = r_dip
    sigma_guess  = (r_outer - r_inner) / 2

    print(f'{disk} gap{gap_ix}:')
    print(f'  gap_dip    = {r_dip:.3f}"   → r_gap  initial guess')
    print(f'  inner_edge = {r_inner:.3f}"  |  outer_edge = {r_outer:.3f}"')
    print(f'  ring_peak  = {r_ring:.3f}"')
    print(f'  fit range  = [{r_min:.3f}", {r_max:.3f}"]')
    print(f'  sigma_gap  ≈ {sigma_guess:.3f}"')

    return r_min, r_max, r_gap_guess, sigma_guess

# preview all available disks/gaps in gap_clicks
print('Disks in gap_clicks:')
print(clicks[clicks['label']=='gap_dip'][['disk','r_arcsec']].to_string(index=False))

In [ ]:
def eqC1(r, T0, q, r_gap, sigma_gap, delta_gap):
    """Eq. C1 brightness temperature model.
    
    Parameters
    ----------
    r          : radial distance in arcsec
    T0         : background normalisation at r=0.1" (K)
    q          : power-law index of background
    r_gap      : gap centre (arcsec)
    sigma_gap  : gap Gaussian width (arcsec)
    delta_gap  : gap depletion factor (multiplicative; delta=1 → no gap)
    """
    background = T0 * (r / 0.1)**(-q)
    Gamma = (delta_gap - 1) * np.exp(-0.5 * ((r - r_gap) / sigma_gap)**2)
    return background / (1 + Gamma)

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────────────
target  = 'J1852'
gap_ix  = 0

# get r_min, r_max, r_gap and sigma_gap from gap_clicks (rough initial guesses)
r_min, r_max, r_gap_guess, sigma_guess = get_gap_guesses(target, gap_ix)

# remaining guesses — read T0 and q by eye from the frank profile plot
# T0: brightness temperature near the gap (K), q: slope (~1-3)
# delta_gap: depth ratio — start with 2-10
p0 = [50., 2.0, r_gap_guess, sigma_guess, 5.0]
#            T0    q     r_gap          sigma     delta
# ────────────────────────────────────────────────────────────────────────────

# load frank profile
frank_file = rf'D:\exoALMA_disk_data\{target}\{target}_frank_profile.txt'
dat = np.loadtxt(frank_file, comments='#')
r, r_au, I_Jysr, Tb = dat[:, 0], dat[:, 1], dat[:, 2], dat[:, 3]

# restrict to fit range
mask = (r >= r_min) & (r <= r_max)
r_fit, Tb_fit = r[mask], Tb[mask]

# fit
popt, pcov = curve_fit(eqC1, r_fit, Tb_fit, p0=p0, maxfev=10000)
T0_fit, q_fit, rgap_fit, sgap_fit, dgap_fit = popt
perr = np.sqrt(np.diag(pcov))

print(f'\nFitted parameters:')
print(f'  T0     = {T0_fit:.1f} ± {perr[0]:.1f} K')
print(f'  q      = {q_fit:.2f} ± {perr[1]:.2f}')
print(f'  r_gap  = {rgap_fit:.4f} ± {perr[2]:.4f} "')
print(f'  s_gap  = {sgap_fit:.4f} ± {perr[3]:.4f} "')
print(f'  d_gap  = {dgap_fit:.2f} ± {perr[4]:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r, Tb, 'k-', lw=1, label='frank profile')
r_model = np.linspace(r_min, r_max, 500)
ax.plot(r_model, eqC1(r_model, *popt), 'r--', lw=1.5, label='Eq. C1 fit')
ax.axvline(rgap_fit, color='r', ls=':', alpha=0.6, label=f'$r_{{gap}}$ = {rgap_fit:.3f}"')
ax.axvspan(r_min, r_max, alpha=0.07, color='blue', label='fit range')
ax.set_xlabel('r (arcsec)')
ax.set_ylabel('$T_b$ (K)')
ax.set_title(f'{target}  gap{gap_ix}')
ax.legend(fontsize=8)
ax.set_xlim([0, r[r < 1.5].max()])
plt.tight_layout()
plt.show()